# Iterator and Collector: fan out, fan in

![Iterator and Collector flow](iterator_collector_flow.png)

The flow has five nodes, left to right:

- **FlowInputNode** takes an *array* of questions
- **PromptIterator** splits the array and sends each item downstream one at a time
- **PromptProcessor** is a Prompt node that runs once per item
- **ResponseCollector** waits for every iteration and reassembles the results into an array
- **FlowOutput** returns the collected array

> **Teaching/Learning Tip:** this is fan-out/fan-in orchestration, the pattern you would otherwise hand-write with Step Functions or queue code, defined declaratively. The node palette also has Conditions, Knowledge Bases, Agents, Guardrails and Lambda nodes for richer flows (the next modules).

## How the data moves through the flow

The flow is a loop. It takes a list, runs the same prompt on each item separately, and returns the answers as a list in the original order. Here is what happens to the three questions this notebook sends:

1. **FlowInputNode** receives the whole array. Its output is typed `Array`, which is why you can't send plain text to this flow.
2. **PromptIterator** splits the array apart. It sends the questions on one at a time through its `arrayItem` output. It also sends the count, 3, through its `arraySize` output. Everything downstream of `arrayItem` runs once per item, so the next node runs three times.
3. **PromptProcessor** is an ordinary prompt node. Its template is just `{{userPrompt}}`, so each question goes to the model as written. It doesn't know it's inside a loop. It sees one question per run and returns one answer.
4. **ResponseCollector** gathers the answers. It has two inputs: each answer (`arrayItem`) and the count from the iterator (`arraySize`). The count tells it how many answers to wait for. Once it has three, it releases them as one array, `collectedArray`.
5. **FlowOutput** returns that array. The code below pairs each answer with its question and prints them.

```
[Q1, Q2, Q3] → Iterator ─┬─ Q1 → Prompt → A1 ─┐
                         ├─ Q2 → Prompt → A2 ─┼→ Collector → [A1, A2, A3]
                         ├─ Q3 → Prompt → A3 ─┘       ↑
                         └─ arraySize = 3 ────────────┘
```

> **Teaching/Learning Tip:** the connection that people usually miss goes straight from the iterator to the collector. It carries `arraySize`. Without it the collector wouldn't know when the loop is finished.

### Why loop instead of sending one big prompt

You could paste all three questions into one prompt. Running them separately has these advantages:

- Each item gets its own clean model call, so answers don't blend together or skip items.
- You get the answers as a list you can index and process, instead of one block of text to parse.
- The flow works for any list length without a change.

In real use, the list might be 50 customer reviews to summarize, a batch of contracts to pull fields from, or product descriptions to translate. Each item goes through the same steps, and you get one result per item.

## Setup

In [ ]:
import sys
sys.path.append("..")  # flow_utils.py lives one folder up
import flow_utils

session, region, account = flow_utils.get_session_info()  # from your current AWS login
agent = session.client("bedrock-agent")            # control plane: create and manage flows
runtime = session.client("bedrock-agent-runtime")  # data plane: invoke flows
iam = session.client("iam")
model_id = flow_utils.default_model_id(region, account)
print(f"Account {account} | region {region} | model {model_id}")

role_arn = flow_utils.ensure_role(iam, region, account)

## Load the template and create the flow

The definition lives in `iterator_collector_flow.json`. It has one placeholder, `$$PROMPT_MODEL_ID`. `load_template` fills it in on the parsed JSON, so a value can never break the JSON or collide with another variable.

In [ ]:
template = flow_utils.load_template("iterator_collector_flow.json", {"$$PROMPT_MODEL_ID": model_id})

flow_id = flow_utils.create_flow(
    agent, template["name"], template["description"], template["definition"], role_arn, template.get("tags"))
version, alias_id = flow_utils.deploy_flow(agent, flow_id)
print("flow_id:", flow_id, "| version:", version, "| alias_id:", alias_id)

## Invoke with an array

> **Teaching/Learning Tip:** because the Input node is typed **Array**, the input must be a JSON array, not plain text. In the console **Test flow** pane you would paste, for example:
> `["What is Amazon Bedrock in one sentence?", "Write a haiku about cloud computing", "Name three AWS regions"]`
> Three prompts go in, the iterator runs each through the model, the collector merges them: one invocation, one result.

In [ ]:
questions = [
    "What is Amazon Bedrock in one sentence?",
    "Write a haiku about cloud computing.",
    "Name three AWS regions.",
]
outputs = flow_utils.invoke_flow(runtime, flow_id, alias_id, questions)

for question, answer in zip(questions, outputs[0]):
    print(f"Q: {question}\nA: {answer}\n")

## Cleanup

Delete the alias, then the version, then the flow. An alias or version blocks deleting what it points to.
The sample IAM role is left in place so the next module can reuse it. Delete it with
`flow_utils.delete_role(iam)` when you are completely done.

In [ ]:
flow_utils.cleanup_flow(agent, flow_id, alias_id, version)